In [21]:
#Packages Used
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from sympy import *
from sympy import symbols, Function

In [22]:
# 1. Geometric Definition (Constant Distances)
# Tower Definitions
Lx_st = 0  # Horizontal distance from substructure center to tower base
Lz_st = 89.92 + 10  # Vertical distance from substructure center of mass to tower base
L_t = 80  # Tower Length

# Turbine Mass
m_Rotor = 1.1e5           # mass of the rotor [kg]
m_Nacelle = 2.4e5         # mass of the nacelle [kg]
m_Tower = 3.47e5          # mass of the tower [kg]
m_t = m_Rotor + m_Nacelle + m_Tower  # total mass of the turbine [kg]

# Physical constants
pi = 3.14159
rho_w = 1025        # seawater density [kg/m^3]
g = 9.81            # gravity [m/s^2]

# Substructure Definitions
m_s = 7.46633e6  # Substructure mass, including ballast
r_s = 4.7  # Substructure Radius, assumed constant throughout
cross_area_s = pi * r_s**2  # Cross-sectional area of the substructure
H_s = 120 + 10  # Spar Buoy Length
rho_s = m_s / (cross_area_s * H_s)  # Substructure density, assuming uniform

# Buoyancy and restoring stiffness
draft_s = (cross_area_s * H_s * rho_s) / (cross_area_s * rho_w)
submerged_volume_s = cross_area_s * draft_s
KBuoyancy_s = draft_s / 2
J_sub = 1/2 * pi * r_s**4
BM_s = J_sub / submerged_volume_s
KGravity_s = H_s / 2
GM_s = KBuoyancy_s + BM_s - KGravity_s

k_phi_s = rho_w * g * submerged_volume_s * GM_s
k_s_z = rho_w * g * cross_area_s
k_phi_st = 1e9

# Time and generalized coordinates
t = symbols('t')
psi_x_s = Function('psi_x_s')(t)
psi_z_s = Function('psi_z_s')(t)
psi_phi_s = Function('psi_phi_s')(t)
psi_phi_t = Function('psi_phi_t')(t)

x_s = psi_x_s
z_s = psi_z_s
phi_s = psi_phi_s
phi_t = psi_phi_t

# Small-angle linearized tower/nacelle positions
x_t = x_s + Lx_st - Lz_st * phi_s - L_t * phi_t
z_t = z_s + Lx_st * phi_s + Lz_st + L_t
phi_st = phi_s - phi_t

# Velocities
x_s_dot = diff(x_s, t)
z_s_dot = diff(z_s, t)
phi_s_dot = diff(phi_s, t)

x_t_dot = diff(x_t, t)
z_t_dot = diff(z_t, t)
phi_t_dot = diff(phi_t, t)

# Kinetic energy
J_s = 4.229e9
T_s = 1/2 * m_s * (x_s_dot**2 + z_s_dot**2) + 1/2 * J_s * phi_s_dot**2
T_t = 1/2 * m_t * (x_t_dot**2 + z_t_dot**2)
T = T_s + T_t

# Potential energy
V_s = m_s * g * z_s + 1/2 * k_s_z * z_s**2 + 1/2 * k_phi_s * phi_s**2
V_t = m_t * g * z_t + 1/2 * k_phi_st * phi_st**2
V = V_s + V_t

# External forces
F_wave = 4.07e5
M_wave = 38468000
F_wind = 3690000

q = [x_s, z_s, phi_s, phi_t]
Q = []
for qi in q:
    Q_i = F_wind * diff(x_t, qi) + F_wave * diff(z_s, qi) + M_wave * diff(phi_s, qi)
    Q.append(simplify(Q_i))

# Lagrangian and equations of motion
L = T - V
EOM_x_s = diff(diff(L, x_s_dot), t) - diff(L, x_s) - Q[0]
EOM_z_s = diff(diff(L, z_s_dot), t) - diff(L, z_s) - Q[1]
EOM_phi_s = diff(diff(L, phi_s_dot), t) - diff(L, phi_s) - Q[2]
EOM_phi_t = diff(diff(L, phi_t_dot), t) - diff(L, phi_t) - Q[3]

unknowns = [diff(psi_x_s, (t, 2)), diff(psi_z_s, (t, 2)), diff(psi_phi_s, (t, 2)), diff(psi_phi_t, (t, 2))]
MTRX = linear_eq_to_matrix([simplify(EOM_x_s), simplify(EOM_z_s), simplify(EOM_phi_s), simplify(EOM_phi_t)], unknowns)
M = simplify(MTRX[0])
F = simplify(MTRX[1])

# Separate F into stiffness terms and external forcing:
# M q'' = F(q, t) => M q'' + K q = F_ext
q_state = [psi_x_s, psi_z_s, psi_phi_s, psi_phi_t]
K_coeffs, F_ext = linear_eq_to_matrix(list(F), q_state)
K = simplify(-K_coeffs)
F_ext = simplify(F_ext)

print('M =')
pprint(M)
print('\nK =')
pprint(K)
print('\nF_ext =')
pprint(F_ext)




M =
⎡ 8163330.0       0       -69644240.0   -55760000.0 ⎤
⎢                                                   ⎥
⎢     0       8163330.0        0             0      ⎥
⎢                                                   ⎥
⎢-69644240.0      0      11187852460.8  5571539200.0⎥
⎢                                                   ⎥
⎣-55760000.0      0      5571539200.0   4460800000.0⎦

K =
⎡0         0                 0                0      ⎤
⎢                                                    ⎥
⎢0  697811.455201275         0                0      ⎥
⎢                                                    ⎥
⎢0         0          90810038.8475192  -1000000000.0⎥
⎢                                                    ⎥
⎣0         0           -1000000000.0    1000000000.0 ⎦

F_ext =
⎡ -3690000  ⎤
⎢           ⎥
⎢79675267.3 ⎥
⎢           ⎥
⎢330236800.0⎥
⎢           ⎥
⎣ 295200000 ⎦
